# FDSI benchmark v1.1: optimisation -- roa (ground-truth upper bound)

Re-runs `../fdsi_benchmark/03c_roa_optimisation.ipynb`'s pooled search with adapt_decomp v1.1, on the same 3
pooled triangular recordings (ramp 40 s, 10 s and 5 s; sub-01, 30 dB) and 50 trials.

Scores every trial directly on mean RoA against ground truth (`100 - roa_mean`, summed across
the pool), the upper-bound reference for what a training loss could reach. With unit
selection, RoA is scored on the selected units only.

What changed from v1.0:

| | v1.0 (`../fdsi_benchmark/`) | v1.1 (this folder) |
|---|---|---|
| Entry point | `optimize_adapt_decomp_pooled_memory(_pareto)` | `optimize_adapt_decomp` |
| Sampler | TPE (independent), 15 startup trials | multivariate TPE, 15 startup trials |
| Searched | `wh_learning_rate`, `sv_learning_rate` (`centroid_momentum` fixed at 0.95) | + `centroid_momentum` in [0, 0.95] |
| `sv_loss` per recording | summed over units | mean over units (`sv_loss_reduction="mean"`) |
| Units adapted and scored | every GT-matched calibration unit | GT-matched units with calibration CoV-ISI <= 0.3 |
| Samples adapted | whole recording, from sample 0 (calibration window re-adapted) | after the 5 s calibration window, source FIFO seeded from its tail |
| Parallelism | `n_jobs=1` | `n_jobs=1`, `n_workers=3` (sequential trials, reproducible) |
| `lr_mode` | `fixed` and `rel_error` | `fixed` only |

Runs the search if no complete cached `study.pkl` exists under
`OUTPUTS_ROOT/adaptation_v1_1/optimisation/lr_fixed/mtpe_roa/` (`RUN_OPTIMISATION=True`); a rerun
only loads.

## Config

In [ ]:
RUN_OPTIMISATION = True   # run the search unless a complete cached study.pkl exists

import sys
from pathlib import Path
sys.path.insert(0, '../..')          # package root
sys.path.insert(0, str(Path.cwd()))  # this folder, for fdsi_v11 (which imports ../fdsi_benchmark/fdsi_common)

import fdsi_v11 as fv
fc = fv.fc

g = fv.Grid.from_yaml(Path('../../configs/data_configs/fdsi_benchmark_grid.yaml'))
BASE_CONFIG_PATH = Path('../../configs/adapt_configs/default_muniverse.yaml')   # v1.0's base config

print(f'{len(g.recordings())} recordings; calibration window [0, {g.cal_end}) samples; '
      f'pool {g.pool_conditions} ({g.optim_sub}, {g.optim_snr} dB); holdout {g.holdout_conditions}')

OBJECTIVES = 'roa'
V10_SEARCH_DIR = 'tpe_roa'
OPT_DIR = g.opt_dir('v1.1', fv.SEARCH_DIRS['roa'])
print('Search settings:', fv.SEARCH)

## Data pool

The same 3 pooled recordings as v1.0 (`configs/data_configs/fdsi_pool_memory_example.yaml`),
restricted to the samples after the calibration window (`fv.calib_end_pool`): with the
source FIFO seeded from the calibration's tail, each trial adapts and is scored from the
calibration end, as `process_from_calib_end` applies it later. The only approximation: the
filters restart at the window's end (a few-ms transient).

Unit selection then keeps the units with calibration CoV-ISI <= 0.3; a recording's dropped
units are neither adapted nor scored during the search (the applied config later adapts
every unit).

In [ ]:
import yaml
import pandas as pd
from adapt_decomp import load_data

with open('../../configs/data_configs/fdsi_pool_memory_example.yaml') as f:
    pool = fv.calib_end_pool(load_data(yaml.safe_load(f)), g.cal_end)
assert set(pool) == set(g.pool_conditions), pool.keys()

cov_th = fv.SEARCH['unit_selection_kwargs']['cov_th']
units_v10 = {name: d.calibration.sources.shape[1] for name, d in pool.items()}
units_v11 = {name: int(d.calibration.unsupervised_mask(cov_th=cov_th).sum()) for name, d in pool.items()}
pd.DataFrame({'units (v1.0 search)': units_v10, f'units (v1.1 search, CoV-ISI <= {cov_th})': units_v11,
              'samples after calibration': {name: d.emg.shape[0] for name, d in pool.items()}})

## Optimisation

In [ ]:
result = fv.run_search(OBJECTIVES, pool, fv.search_base_config(BASE_CONFIG_PATH), OPT_DIR,
                       g.n_trials, RUN_OPTIMISATION, g.tol_spike_ms)
study = result.study
if 'wall_time_s' in study.user_attrs:
    print(f"Search wall time: {study.user_attrs['wall_time_s'] / 60:.1f} min")

In [ ]:
chosen = {'roa': study.best_trial}
best_configs = {'roa': result.best_config}
print(f'Best trial {study.best_trial.number}: value {study.best_value:.4g}')

### v1.0's search, for comparison

The cached v1.0 study (`lr_fixed/tpe_roa`) and its selected trial.

In [ ]:
from adapt_decomp.adaptation.optimize import _select_min_sv_loss

old_study = fv.load_study(g.opt_dir('v1.0', V10_SEARCH_DIR))
old_chosen = old_study.best_trial
print(f'v1.0: {len(old_study.trials)} trials, selected trial {old_chosen.number}')

### Optimisation landscape -- RoA vs pooled losses, v1.1 vs v1.0

RoA (colour/y) is comparable across rows; the loss axes are not: v1.1's `sv_loss` is a
per-unit mean scored after the calibration window, v1.0's a per-recording sum over the whole
recording.

In [ ]:
from adapt_decomp.utils.plots import plot_optimisation_landscape_grid, plot_pareto_scatter_grid

first = next(iter(chosen.values()))
fig = plot_optimisation_landscape_grid({'v1.1': study, 'v1.0': old_study},
                                       {'v1.1': 'v1.1 (this search)', 'v1.0': 'v1.0'},
                                       {'v1.1': first, 'v1.0': old_chosen})
fig.show()

### Parameters vs pooled RoA

Each completed trial's parameters against its pooled mean RoA (colour: pooled `sv_loss`),
including `centroid_momentum`, which v1.0 kept fixed at 0.95.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

trials = study.trials_dataframe(attrs=('number', 'params', 'user_attrs', 'state'))
trials = trials[trials['state'] == 'COMPLETE']
params = [c for c in trials.columns if c.startswith('params_')]
fig, axs = plt.subplots(1, len(params), figsize=(5 * len(params), 4), sharey=True)
for ax, param in zip(axs, params):
    sns.scatterplot(data=trials, x=param, y='user_attrs_roa_mean_pooled', hue='user_attrs_sv_loss',
                    palette='viridis_r', ax=ax, legend=ax is axs[-1])
    if 'learning_rate' in param:
        ax.set_xscale('log')
    for trial in chosen.values():
        ax.axvline(trial.params[param.removeprefix('params_')], color='r', ls='--', lw=1)
    ax.set(xlabel=param.removeprefix('params_'), ylabel='pooled mean RoA (%)')
plt.suptitle('v1.1 trials (dashed: selected)')
plt.tight_layout()
plt.show()

### Unit-count bias of the pooled objective

Each pooled recording's share of the pooled `sv_loss`, averaged over trials, against the
units the search scored on it. In v1.0 (sum over units) the share tracks the unit count, so
recordings with more units steer the search; with v1.1's per-unit mean every recording weighs
alike.

In [ ]:
shares = pd.concat([fv.pooled_loss_shares(old_study, units_v10).assign(version='v1.0'),
                    fv.pooled_loss_shares(study, units_v11).assign(version='v1.1')])
share_summary = (shares.groupby(['version', 'recording'])
                 .agg(units=('units', 'first'), mean_share=('share', 'mean')).round(3))
display(share_summary)
sns.barplot(data=shares, x='recording', y='share', hue='version')
plt.ylabel('share of pooled sv_loss')
plt.show()

### Trial wall time

Median seconds per trial (the first v1.1 trial also starts the worker processes). v1.1 adapts
only after the calibration window and spreads each trial's recordings over 3 processes.

In [ ]:
durations = pd.DataFrame({'v1.0': fv.trial_durations(old_study).describe(),
                          'v1.1': fv.trial_durations(study).describe()})
durations.loc[['count', 'mean', '50%', 'min', 'max']].round(2)

## Promote the winning config

Saves the winner as a reusable `configs/adapt_configs/`
entry (`optim_muniverse_fdsi_v11_roa.yaml`), which
`04c` applies to the full grid, next to v1.0's promoted winner.

In [ ]:
for branch, config in best_configs.items():
    print(f'Promoted {branch} -> {fv.promote(config, branch)}')
fv.hyperparameter_table(list(best_configs))